In [ ]:
# =============================================================
# AI Stock Market Research Assistant — Config Setup
# File    : pipeline/00_setup_config.py
# Purpose : Creates and manages the ticker configuration table
#           in Unity Catalog. Run this ONCE to initialize,
#           then use to add/remove tickers anytime.
# =============================================================

# Databricks notebook source


## Pipeline Config — Ticker Registry
Stores the list of tickers to track in Unity Catalog.
Edit this notebook to add/remove tickers without touching
the ingestion pipeline.


In [ ]:
from pyspark.sql import SparkSession
from datetime import datetime

spark = SparkSession.builder.getOrCreate()


In [ ]:
# ------------------------------------------------------------------
# 1. Create config schema in Unity Catalog
# ------------------------------------------------------------------
spark.sql("CREATE SCHEMA IF NOT EXISTS main.config")
print("Schema main.config ready")


In [ ]:
# ------------------------------------------------------------------
# 2. Create ticker_config table
# ------------------------------------------------------------------
spark.sql("""
    CREATE TABLE IF NOT EXISTS main.config.ticker_config (
        ticker      STRING  NOT NULL,
        name        STRING,
        sector      STRING,
        active      BOOLEAN NOT NULL DEFAULT true,
        added_at    TIMESTAMP,
        notes       STRING
    )
    USING DELTA
    COMMENT 'Registry of tickers tracked by the ingestion pipeline.
             Set active=false to exclude a ticker without deleting it.'
""")
print("Table main.config.ticker_config ready")


In [ ]:
# ------------------------------------------------------------------
# 3. Seed initial tickers
#    Start with 5 for testing — add more when ready
#    Change active=True/False to include/exclude per run
# ------------------------------------------------------------------

# Clear and re-seed (idempotent — safe to re-run)
spark.sql("DELETE FROM main.config.ticker_config")

tickers = [
    # --- ACTIVE (included in every pipeline run) ---
    ("AAPL",  "Apple Inc.",             "Technology",  True,  "Initial test set"),
    ("MSFT",  "Microsoft Corp.",        "Technology",  True,  "Initial test set"),
    ("GOOGL", "Alphabet Inc.",          "Technology",  True,  "Initial test set"),
    ("NVDA",  "NVIDIA Corp.",           "Technology",  True,  "Initial test set"),
    ("META",  "Meta Platforms Inc.",    "Technology",  True,  "Initial test set"),

    # --- INACTIVE (excluded until you set active=True below) ---
    ("JPM",   "JPMorgan Chase & Co.",   "Finance",     False, "Expand later"),
    ("BAC",   "Bank of America Corp.",  "Finance",     False, "Expand later"),
    ("GS",    "Goldman Sachs Group.",   "Finance",     False, "Expand later"),
    ("MS",    "Morgan Stanley",         "Finance",     False, "Expand later"),
    ("V",     "Visa Inc.",              "Finance",     False, "Expand later"),
    ("JNJ",   "Johnson & Johnson",      "Healthcare",  False, "Expand later"),
    ("UNH",   "UnitedHealth Group",     "Healthcare",  False, "Expand later"),
    ("PFE",   "Pfizer Inc.",            "Healthcare",  False, "Expand later"),
    ("ABBV",  "AbbVie Inc.",            "Healthcare",  False, "Expand later"),
    ("MRK",   "Merck & Co.",            "Healthcare",  False, "Expand later"),
    ("XOM",   "Exxon Mobil Corp.",      "Energy",      False, "Expand later"),
    ("CVX",   "Chevron Corp.",          "Energy",      False, "Expand later"),
    ("AMZN",  "Amazon.com Inc.",        "Consumer",    False, "Expand later"),
    ("TSLA",  "Tesla Inc.",             "Consumer",    False, "Expand later"),
    ("WMT",   "Walmart Inc.",           "Consumer",    False, "Expand later"),
]

now = datetime.now().isoformat()
rows = [(t, n, s, a, now, note) for t, n, s, a, note in tickers]

df = spark.createDataFrame(rows,
    ["ticker", "name", "sector", "active", "added_at", "notes"])
df.write.mode("append").saveAsTable("main.config.ticker_config")

print(f"Seeded {len(rows)} tickers")


In [ ]:
# ------------------------------------------------------------------
# 4. Show current config
# ------------------------------------------------------------------
print("\n=== Active tickers (will be ingested) ===")
spark.sql("""
    SELECT ticker, name, sector, notes
    FROM main.config.ticker_config
    WHERE active = true
    ORDER BY sector, ticker
""").show(truncate=False)

print("\n=== Inactive tickers (skipped) ===")
spark.sql("""
    SELECT ticker, name, sector, notes
    FROM main.config.ticker_config
    WHERE active = false
    ORDER BY sector, ticker
""").show(truncate=False)


## How to activate more tickers

Run this SQL in the cell below to activate all Finance tickers:
```sql
UPDATE main.config.ticker_config
SET active = true
WHERE sector = 'Finance'
```

Or activate a specific ticker:
```sql
UPDATE main.config.ticker_config
SET active = true
WHERE ticker = 'JPM'
```

Or activate ALL tickers at once:
```sql
UPDATE main.config.ticker_config
SET active = true
```


In [ ]:
# ------------------------------------------------------------------
# 5. Helper query — use this in any notebook to load active tickers
# ------------------------------------------------------------------
print("=== Load active tickers in any notebook ===")
print("""
TICKERS = [
    row.ticker for row in
    spark.sql("SELECT ticker FROM main.config.ticker_config WHERE active = true ORDER BY ticker")
    .collect()
]
print(f"Loaded {len(TICKERS)} active tickers: {TICKERS}")
""")
